# Programming for Linguistics - Lesson 5: N-grams and Frequency Analysis

In this lesson, we will explore **N-grams**, a fundamental concept in Computational Linguistics and Natural Language Processing (NLP). 

An **N-gram** is a contiguous sequence of *n* items from a given sample of text or speech. 
*   **Unigrams**: Single words (`n=1`)
*   **Bigrams**: Pairs of consecutive words (`n=2`)
*   **Trigrams**: Triplets of consecutive words (`n=3`)

N-grams are used for various tasks, such as:
*   **Language Modeling**: Predicting the next word in a sentence (like autocomplete).
*   **Collocation Extraction**: Identifying common phrases or idioms.
*   **Author Attribution**: Analyzing the unique stylistic patterns of an author.

---


## Step 1: Environment Setup

To begin, we need to import some utility functions and ensure our Python environment can find them. We add the project directory to `sys.path` so that our custom modules (like `utils.py`) can be imported properly.


In [4]:
from utils import infile_outsents_comp, get_bigrams_freqs

In [3]:
import sys
sys.path.append('/Users/luca/Gits/pfl_2026_unicatt/')

In [5]:
sys.path

['/opt/homebrew/Cellar/python@3.11/3.11.13/Frameworks/Python.framework/Versions/3.11/lib/python311.zip',
 '/opt/homebrew/Cellar/python@3.11/3.11.13/Frameworks/Python.framework/Versions/3.11/lib/python3.11',
 '/opt/homebrew/Cellar/python@3.11/3.11.13/Frameworks/Python.framework/Versions/3.11/lib/python3.11/lib-dynload',
 '',
 '/Users/luca/Envs/stanza/lib/python3.11/site-packages',
 '/Users/luca/Envs/stanza/lib/python3.11/site-packages/textcomplexity-0.11.0-py3.11.egg',
 '/Users/luca/Envs/stanza/lib/python3.11/site-packages/nltk_tgrep-1.0.6-py3.11.egg',
 '/Users/luca/Gits/pfl_2026_unicatt/']

## Step 2: Loading the Corpus

We load the text of Bram Stoker's *Dracula* (Project Gutenberg ID 345). 

The function `infile_outsents_comp` is a custom utility that:
1. Reads the raw text file.
2. Normalizes the content.
3. Splits it into a list of sentences, where each sentence is a **list of tokens** (words and punctuation).


In [7]:
text = infile_outsents_comp('assets/pg345.txt')
# text

## Step 3: The "Zip" Trick for N-grams

Generating N-grams manually with loops can be complex. Python provides a very elegant way to do this using the `zip()` function and **list slicing**.

### How it works:
If we have a list `a = [1, 2, 3, 4]`, then `a[1:]` is `[2, 3, 4]`. 
When we `zip(a, a[1:])`, Python pairs them up:
*   `1` with `2` 
*   `2` with `3` 
*   `3` with `4` 

This creates our bigrams! For trigrams, we would zip `a`, `a[1:]`, and `a[2:]`.


In [8]:
a = 'a b c d e f g'.split()
b = "1 2 3 4 5 6 7".split()
len(b) == len(a)

for j,k in zip(a, a[1:]):
    print(j,k)

a b
b c
c d
d e
e f
f g


In [10]:
sent = text[34]

#Ngrams base 2 or bigrams
for w1,w2, w3 in zip(sent, sent[1:], sent[2:] ):
    print(w1, w2, w3)

## Step 4: Building a Frequency Analyzer

Now that we can generate bigrams, we want to see which ones are the most common in *Dracula*. However, many frequent bigrams are not very informative (e.g., "of the", "in a"). These often involve **stopwords** (highly frequent words with little semantic value) or **punctuation**.

### Logic of `get_bigrams_freqs`:
1.  **Filtering**: We only keep bigrams where *neither* word is punctuation or a stopword.
2.  **Counting**: We use a `defaultdict(int)`. This is a special dictionary that automatically initializes a new key with `0` if it doesn't exist, making counting much cleaner.
3.  **Sorting**: We sort the resulting dictionary by frequency in descending order.


In [24]:
# typewriter
# machine gun

In [84]:
from collections import defaultdict
from string import punctuation
punctuation += "”“’--"
from nltk.corpus import stopwords
en_stops = stopwords.words('english')

def get_bigrams_freqs(corpus, threshold= 5):
    # populate the dictionary with bigrams frequencies, where bigrams are keys and their value is the frequency
    # out = {}
    out = defaultdict(int)
    for text in corpus:
        for w1,w2 in zip(text, text[1:]):
            if w1 not in punctuation and w2 not in punctuation and w1 not in en_stops and w2 not in en_stops:
                bigram = f'{w1}_{w2}'
                # if bigram in out:
                #     out[bigram] += 1
                # else:
                #     out[bigram] = 1
                out[bigram] +=1
    sorted_freqs = sorted(out.items(), key=lambda x : x[1], reverse=True )
    return [t for t in sorted_freqs if t[1] >= threshold]


get_bigrams_freqs(text, threshold=2)



[('van_helsing', 315),
 ('could_see', 87),
 ('madam_mina', 87),
 ('dr._seward', 84),
 ('project_gutenberg', 71),
 ('lord_godalming', 66),
 ('mrs._harker', 64),
 ('dr._van', 58),
 ('friend_john', 57),
 ('last_night', 46),
 ('mr._morris', 37),
 ('poor_lucy', 33),
 ('poor_dear', 31),
 ('miss_lucy', 30),
 ('dear_madam', 26),
 ('jonathan_harker', 24),
 ('quincey_morris', 24),
 ('could_hear', 23),
 ('must_go', 22),
 ('came_back', 22),
 ('let_us', 21),
 ('old_man', 20),
 ('mrs._westenra', 20),
 ('said_van', 18),
 ('united_states', 17),
 ('mr._hawkins', 17),
 ('thank_god', 17),
 ('helsing_said', 17),
 ('_czarina_catherine_', 17),
 ('project_gutenberg™', 17),
 ('electronic_works', 16),
 ('shall_try', 15),
 ('shall_go', 15),
 ('dear_lucy', 15),
 ('go_back', 15),
 ('poor_fellow', 14),
 ('said_nothing', 14),
 ('went_back', 13),
 ('come_back', 13),
 ('amongst_us', 13),
 ('came_away', 13),
 ('_mina_harker', 13),
 ('gutenberg_literary', 13),
 ('literary_archive', 13),
 ('journal_chapter', 12),
 ('cou

### Cleaning Data: Punctuation and Stopwords

To get meaningful results, we often need to expand the default list of punctuation provided by Python to include "smart" quotes and other symbols found in literary texts.


In [63]:
from string import punctuation
punctuation += "”“’"
punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~”“’'

## Step 5: Advanced Python - Lambda and Sorting

Dictionaries in Python are not sorted by default. To sort them by their **values** (the frequencies), we use the `sorted()` function.

We need to tell Python: "Don't sort by the word (the key), sort by the number (the value)". 

A **Lambda function** is a small, anonymous function defined in one line. For example, `lambda x: x[1]` tells Python to look at the second element of each pair (the frequency) when sorting.


In [54]:
d = {"a" : 345, 
     "b" : 846,
     "c" : 23}
def sorting_ints(tup):
    return tup[1]

sorted(d.items(), key=lambda x : x[1], reverse=True )


[('b', 846), ('a', 345), ('c', 23)]

In [53]:
lambda_ints = lambda x : x[1]
lambda_ints(("a", 754))

754

In [37]:
d = defaultdict(int)
# d = {}
d["of"] +=1
# d.items()

KeyError: 'of'

In [41]:
vocab = set()
for sent in text:
    vocab.update(sent)
vocab = {word:i for i, word in enumerate(vocab)}
vocab

{'rich': 0,
 'nj': 1,
 'not-so-far-off': 2,
 'denial': 3,
 'ships': 4,
 'experimentally': 5,
 'pistol': 6,
 'bidding': 7,
 'a-': 8,
 'authors': 9,
 'pursue': 10,
 'soothingly': 11,
 'ravings': 12,
 'accepting': 13,
 'crisis': 14,
 'despairingly': 15,
 'boiler': 16,
 'concealed': 17,
 'zoölogical': 18,
 'compel': 19,
 'beneficial': 20,
 'downloading': 21,
 'seconds': 22,
 'twenty-one': 23,
 'kitchen': 24,
 'experiments': 25,
 'mixes': 26,
 'leisure': 27,
 'whistles': 28,
 'clumps': 29,
 'jamaica': 30,
 'death-chamber': 31,
 'loosened': 32,
 'waltz': 33,
 'eight.': 34,
 'furnace': 35,
 'afflicted': 36,
 'stable': 37,
 'vice': 38,
 'oblivious': 39,
 'camera_': 40,
 'caricaturists': 41,
 'man-stature': 42,
 'gunwale': 43,
 'hearts': 44,
 'gown': 45,
 'anywhere.': 46,
 'materialise': 47,
 'forehead': 48,
 'hall': 49,
 'winter-suffield': 50,
 'engines': 51,
 'drouth': 52,
 'replacement': 53,
 'solved': 54,
 'companion': 55,
 'mysteriously': 56,
 'fashioned': 57,
 'owe': 58,
 'foolhardiness':

In [21]:
sample = text[45][:15]
v = set(sample)
len(v)

13

In [29]:
m = [[0,0,0,0,0,0,0,0,0,0,0,0,0],
     [0,0,0,0,0,0,0,0,0,0,0,0,0],
     [0,0,0,0,0,0,0,0,0,0,0,0,0], 
]

In [35]:
m[0][2] +=1

In [39]:
{word:i for i,word in enumerate(v)}   

{'saxons': 0,
 'there': 1,
 'in': 2,
 ':': 3,
 'the': 4,
 'transylvania': 5,
 'south': 6,
 'nationalities': 7,
 'of': 8,
 'distinct': 9,
 'population': 10,
 'four': 11,
 'are': 12}

In [72]:
def co_occurrence_matrix(txt):
    vocab = set()
    for sent in txt:
        vocab.update(sent)
    vocab = {word:i for i, word in enumerate(vocab)}
    mtx = np.zeros((len(vocab), len(vocab)))
    for sent in txt:
        for w1,w2 in zip(sent, sent[1:]):
            iw1 = vocab[w1]
            iw2 = vocab[w2]
            mtx[iw1,iw2] +=1
    return mtx

m = co_occurrence_matrix(text)
sum(m[162])
    

1.0

In [47]:
import numpy as np

np.array((1,2))

array([1, 2])

In [56]:
zeros_array = np.zeros((3, 5))
zeros_array[1,2] = 10
zeros_array

array([[ 0.,  0.,  0.,  0.,  0.],
       [ 0.,  0., 10.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.]])

In [50]:
np.ones((3, 5))

array([[1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.]])